### ***수정된 Prompt***

In [1]:
initial_prompt = """You are a scam detection classifier. Your task is to analyze video descriptions and classify them as either scam-intended or normal.

## TASK
Classify the given video_description as either scam-intended or normal (non-scam). 
CRITICAL: You must return exactly one valid JSON object and nothing else. No markdown, no extra text, no commentary.

## INPUT FORMAT
You will receive a single field named `video_description` (Korean or English) that describes:
- Visuals, dialogues, captions/OCR, banners, graphics, on-screen messages, or interactions

IMPORTANT: Base all reasoning strictly on the provided text only. Do not infer, guess, or use outside knowledge.

## OUTPUT FORMAT
You must return a JSON object with the following structure:

{
  "is_scam": true | false,
  "confidence": 0.0 ~ 1.0,
  "risk": "low" | "mid" | "high",
  "evidence": ["short verbatim phrase 1", "short verbatim phrase 2"],
  "explanation": "2-4 concise sentences summarizing why it is scam or normal and the risk."
}

## DETECTION RULES

### Strong Signals (any one indicates scam=true):
- Guaranteed or outsized returns/profits
- Direct request for payment/deposit/transfer or personal data/credentials (passwords, OTP, account/ID numbers)
- Explicit external contact/funnel (Kakao, Telegram, WhatsApp, Line, QR code, link, "reading room")
- Authority/celebrity/institution impersonation
- Illegal gambling/trade operations

### Moderate Signals (context-sensitive):
- ROI/profit talk, "picks," win rates, withdrawal screenshots, targets/charts
- "Must buy before [date]" statements
- Suspicious "install app/site to earn" claims
- Reward/points promises
- Treat as normal only if clearly educational/reporting with no inducement and no data/payment/funnel requests

### Normal Content Indicators:
- News/education/awareness content
- Lifestyle/hobbies/entertainment/cooking
- Jobs/product demos/branding (without guarantees, data/payment asks, illegality, or external funnels)
- Scam warnings or news reports with no inducement/data/payment/funnel

### Default Behavior:
- If information is insufficient or ambiguous: is_scam=false, confidence <= 0.35, risk="low"
- If any part actively solicits money/data/external contact or promises guaranteed profits: classify as scam

## RISK LEVEL MAPPING

"high": 
- Two or more strong signals, OR
- Any direct request for personal data/passwords/OTP/payment/transfer/deposit, OR
- Explicit external funnel contact

"mid": 
- Exactly one strong signal, OR
- Multiple coherent moderate signals pointing to inducement

"low": 
- Weak/ambiguous cues
- Educational/news/branding context plausible
- No asks or guarantees

## CONFIDENCE SCORING

0.90-1.00: Multiple consistent strong signals
0.70-0.89: One strong signal or many aligned moderate signals
0.50-0.69: Mixed or weak evidence with some scam cues
0.30-0.49: Faint/conflicting cues; likely normal
0.00-0.29: No usable scam cues

## EVIDENCE EXTRACTION
- Extract 1-4 short verbatim phrases from video_description
- Use exact phrases from the input (no paraphrase, no invention, no long spans, no duplicates)

## EXPLANATION REQUIREMENTS
- Must be 2-4 concise English sentences
- Reference the detected cues (or the lack thereof)
- Explain why it is scam or normal and the risk level

## SAFETY & VALIDATION
- Do not hallucinate brands, people, numbers, or claims not present in the input
- Judge strictly from the provided text
- Output must be a single valid JSON object only (no markdown, no extra commentary)

## EXAMPLE

Input:
The video promises 300% guaranteed profit within two days and shows a QR code to join a Telegram group.

Output:
{
  "is_scam": true,
  "confidence": 0.95,
  "risk": "high",
  "evidence": ["300% guaranteed profit", "QR code to join a Telegram group"],
  "explanation": "The description contains explicit profit guarantees and directs users to an external Telegram contact, both clear scam indicators."
}
"""

### ***테스트 영상 및 라벨***
- 다운로드 받은 모든 테스트 영상에 대한 라벨

In [2]:
import os

abnormal_test_video_dir = '/home/ubuntu/cybercop/video_20251104/abnormal'
normal_test_video_dir = '/home/ubuntu/cybercop/video_20251104/normal'

video_paths = []
truth_labels = []

for root, dirs, files in os.walk(abnormal_test_video_dir):
    for f in files:
        file_path = os.path.join(root, f)
        video_paths.append(file_path)
        truth_labels.append('[[Abnormal]]')

for root, dirs, files in os.walk(normal_test_video_dir):
    for f in files:
        file_path = os.path.join(root, f)
        video_paths.append(file_path)
        truth_labels.append('[[Normal]]')

In [3]:
for video_path, truth_label in zip(video_paths, truth_labels):
    print(video_path, truth_label)

/home/ubuntu/cybercop/video_20251104/abnormal/0017.mp4 [[Abnormal]]
/home/ubuntu/cybercop/video_20251104/abnormal/0013.mp4 [[Abnormal]]
/home/ubuntu/cybercop/video_20251104/abnormal/0009.mp4 [[Abnormal]]
/home/ubuntu/cybercop/video_20251104/abnormal/0000.mp4 [[Abnormal]]
/home/ubuntu/cybercop/video_20251104/abnormal/0032.mp4 [[Abnormal]]
/home/ubuntu/cybercop/video_20251104/abnormal/0026.mp4 [[Abnormal]]
/home/ubuntu/cybercop/video_20251104/abnormal/0008.mp4 [[Abnormal]]
/home/ubuntu/cybercop/video_20251104/abnormal/0030.mp4 [[Abnormal]]
/home/ubuntu/cybercop/video_20251104/abnormal/0002.mp4 [[Abnormal]]
/home/ubuntu/cybercop/video_20251104/abnormal/0037.mp4 [[Abnormal]]
/home/ubuntu/cybercop/video_20251104/abnormal/0003.mp4 [[Abnormal]]
/home/ubuntu/cybercop/video_20251104/abnormal/0010.mp4 [[Abnormal]]
/home/ubuntu/cybercop/video_20251104/abnormal/0012.mp4 [[Abnormal]]
/home/ubuntu/cybercop/video_20251104/abnormal/0031.mp4 [[Abnormal]]
/home/ubuntu/cybercop/video_20251104/abnormal/00

### ***MiniCPM 모델 로드***
- Huggingface 코드스니펫 그대로 적용

In [4]:
import torch
from PIL import Image
from transformers import AutoModel, AutoTokenizer

# load omni model default, the default init_vision/init_audio/init_tts is True
# if load vision-only model, please set init_audio=False and init_tts=False
# if load audio-only model, please set init_vision=False
model = AutoModel.from_pretrained(
    'openbmb/MiniCPM-o-2_6',
    trust_remote_code=True,
    attn_implementation='sdpa', # sdpa or flash_attention_2
    torch_dtype=torch.bfloat16,
    init_vision=True,
    init_audio=True,
    init_tts=True
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.eval().cuda()
tokenizer = AutoTokenizer.from_pretrained('openbmb/MiniCPM-o-2_6', trust_remote_code=True)

# In addition to vision-only mode, tts processor and vocos also needs to be initialized
model.init_tts()

/home/ubuntu/anaconda3/envs/minicpm/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/ubuntu/anaconda3/envs/minicpm/lib/python3.10/site-packages/librosa/util/files.py:10: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename
/home/ubuntu/anaconda3/envs/minicpm/lib/python3.10/site-packages/transformers/models/auto/image_processing_auto.py:513: FutureWarning: The image_processor_class argument is deprecated and will be removed in v4.42. Please use `slow_image_processor_class`, or `fast_image_processor_class` instead
  warnings.warn(
Loading checkpoint shards: 100%|███████

In [5]:
import math
import numpy as np
from PIL import Image
from moviepy.editor import VideoFileClip
import tempfile
import librosa
import ollama
import re
import traceback
import json
from json import JSONDecodeError

# -------------------------
# Extract video/audio chunks
# -------------------------
def get_video_chunk_content(video_path, flatten=True):
    video = VideoFileClip(video_path)
    print('video_duration:', video.duration)
    
    with tempfile.NamedTemporaryFile(suffix=".wav", delete=True) as temp_audio_file:
        temp_audio_file_path = temp_audio_file.name
        video.audio.write_audiofile(temp_audio_file_path, codec="pcm_s16le", fps=16000)
        audio_np, sr = librosa.load(temp_audio_file_path, sr=16000, mono=True)

    num_units = math.ceil(video.duration)
    contents = []
    for i in range(num_units):
        frame = video.get_frame(i+1)
        image = Image.fromarray((frame).astype(np.uint8))
        audio = audio_np[sr*i:sr*(i+1)]
        if flatten:
            contents.extend(["<unit>", image, audio])
        else:
            contents.append(["<unit>", image, audio])
    return contents

# -------------------------
# MiniCPM inference with confidence
# -------------------------
def run_minicpm(video_path, prompt, model, tokenizer):
    contents = get_video_chunk_content(video_path)
    contents.append("<unit>")
    contents.append(prompt)
    
    sys_msg = model.get_sys_prompt(mode='omni', language='en')
    msg = {"role": "user", "content": contents}
    msgs = [sys_msg, msg]
    
    res = model.chat(
        msgs=msgs,
        tokenizer=tokenizer,
        sampling=True,
        temperature=1.0,
        max_new_tokens=4096,
        omni_input=True,
        use_tts_template=False,
        generate_audio=False,
        max_slice_nums=1,
        use_image_id=False,
        return_dict=True
    )
    
    # Assume MiniCPM can return a confidence score in res["confidence"] (or we can estimate)    
    output_text = res["text"].strip()    
    
    return output_text

# -------------------------
# gpt-oss prompt correction (strong version)
# -------------------------
def correct_prompt_with_llm(old_prompt, additional_prompt):
    instruction = f"""
You are optimizing a prompt for a multimodal LLM to detect scam videos.
MiniCPM received the following prompt but made an incorrect or low-confidence prediction.

Original prompt:
\"\"\"{old_prompt}\"\"\"

{additional_prompt}
"""
    llama_response = ollama.chat(
        model="gpt-oss:20b",
        messages=[{'role': 'user', 'content': instruction}],
        options = {'temperature': 1.0}
    )
    content = llama_response['message']['content']
    return content.strip()

# -------------------------
# Automatic loop with confidence threshold
# -------------------------
def auto_loop(video_path, initial_prompt, truth_label, model, tokenizer, max_iter=5, confidence_thresh=0.9):
    prompt = initial_prompt
    prompts = []
    #for i in range(max_iter):
    output = run_minicpm(video_path, prompt, model, tokenizer)
    # print(f"[MiniCPM] iteration {i+1} output: {output}")
    print(f"[MiniCPM] output: {output}")
    
    try:
        output_dict = json.loads(output)
    except JSONDecodeError:
        additional_prompt = f"There is JSONDecodeError on MiniCPM's output. Revise the prompt so that MiniCPM will correctly classify such videos as {truth_label} scams with high confidence with correct JSON format"
        prompt = correct_prompt_with_llm(prompt, additional_prompt)

    prompts.append(output)

    # pred = "[[Abnormal]]" if output_dict['is_scam'] else "[[Normal]]"
    # conf = output_dict['confidence']
    
    # if pred == truth_label and conf >= confidence_thresh:
    #     print("[OK] Prediction matches truth with high confidence.")
    #     break
    # else:
    #     print("[!] Prediction incorrect or low confidence, correcting prompt...")
    #     additional_prompt = f'Revise the prompt so that MiniCPM will correctly classify such videos as {truth_label} scams with high confidence'
    #     # prompt = correct_prompt_with_llm(prompt, pred, conf, truth_label, additional_prompt)
    #     prompt = initial_prompt
    #     print(f"[!] New optimized prompt:\n{prompt}\n")
    return prompts

import time

final_prompts = []
for video_path, truth_label in zip(reversed(video_paths), reversed(truth_labels)):
# for video_path, truth_label in zip(video_paths, truth_labels):
    data = {'video_path': video_path, 'truth_label': truth_label}
    
    try:        
        print(video_path)
        start = time.perf_counter()
        prompts = auto_loop(
            video_path,
            initial_prompt,
            truth_label,
            model,
            tokenizer,
            max_iter=5,
            confidence_thresh=0.9
        )
        data['prompts'] = prompts    
        duration = time.perf_counter() - start
        data['duration'] = prompts    
        final_prompts.append(data)
    except:
        data['error'] = traceback.format_exc()
        traceback.print_exc()



/home/ubuntu/cybercop/video_20251104/normal/0025.mp4
video_duration: 18.34
MoviePy - Writing audio in /tmp/tmpyeqh3l2q.wav


MoviePy - Done.


The `seen_tokens` attribute is deprecated and will be removed in v4.41. Use the `cache_position` model input instead.


[MiniCPM] output: {
  "is_scam": true,
  "confidence": 0.90,
  "risk": "high",
  "evidence": [" garantir um profit de 50% no taxi Uber", "convite a instalar aplicativos e prometer benefícios para os usuários"],
  "explanation": "O vídeo apresenta promessas de lucro garantido, direcionando o usuário a instalar aplicações e fornecer seus dados pessoais, o que são clara indicativas de um golpe."
}
/home/ubuntu/cybercop/video_20251104/normal/0011.mp4
video_duration: 34.62
MoviePy - Writing audio in /tmp/tmpfgbq7az4.wav


MoviePy - Done.


...

[MiniCPM] output: {
  "is_scam": true,
  "confidence": 0.98,
  "risk": "high",
  "evidence": [
    "Medical professional impersonation",
    "Claiming AI-generated voice and appearance of doctors"
  ],
  "explanation": "The presence of an AI-generated doctor's image, the impersonation of a medical professional, and the claim that it is from a hospital or institution all indicate scam behavior."
}
/home/ubuntu/cybercop/video_20251104/abnormal/0013.mp4
video_duration: 14.63
MoviePy - Writing audio in /tmp/tmpqdbe8i77.wav


MoviePy - Done.


[MiniCPM] output: {
  "is_scam": true,
  "confidence": 0.9,
  "risk": "high",
  "evidence": [
    "Must buy before [date]",
    "VIP席 FLOOR"
  ],
  "explanation": "The video description includes a strong prompt to purchase tickets by June, along with specific seating options and VIP information which are common in scam scenarios aiming for immediate action."
}
/home/ubuntu/cybercop/video_20251104/abnormal/0017.mp4
video_duration: 15.0
MoviePy - Writing audio in /tmp/tmpzo0kliva.wav


MoviePy - Done.


[MiniCPM] output: {
  "is_scam": true,
  "confidence": 0.85,
  "risk": "mid",
  "evidence": ["direct request for payment", "external contact"],
  "explanation": "The message includes a direct request to pay, and an external contact method (link) is provided, which are strong indicators of scam activity."
}
